# 29장 · 체크포인터와 thread_id를 이용한 대화 저장

체크포인터는 그래프 상태를 저장합니다. 다음 호출이 어떤 대화를 이어갈지는 thread_id로 구분합니다.

대출과 반납 질문을 같은 대화에서 처리한 다음 새 대화에서 다시 실행합니다. 메시지 병합과 체크포인터의 역할을 나누고 thread_id별로 복원된 기록을 대조합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12, langgraph 1.2.16, langgraph-checkpoint 4.3.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 체크포인터로 같은 스레드의 상태 복원

체크포인터는 그래프가 실행되는 동안 상태를 보관합니다. thread_id가 어느 대화의 상태를 읽고 쓸지 가리킵니다. 노트북에서는 그래프 하나와 메모리 저장소 하나를 만들어 여러 번 사용합니다.


### 셀 01 · thread_id별 초기 상태

제공 함수는 가상 답변을 만들어 실제 LangGraph의 InMemorySaver에 저장합니다. 그래프는 이 셀에서 한 번만 만드세요.


In [1]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langgraph.graph.message import add_messages
from uuid import uuid4
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def respond(state: State) -> dict:
    humans = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not humans:
        raise ValueError("사용자 메시지가 필요합니다.")
    content = f"이 대화의 사용자 발화 {len(humans)}개. 최근 질문: {humans[-1].content}"
    return {"messages": [AIMessage(content=content, id=str(uuid4()))]}

def make_graph():
    builder = StateGraph(State)
    builder.add_node("respond", respond)
    builder.add_edge(START, "respond")
    builder.add_edge("respond", END)
    return builder.compile(checkpointer=InMemorySaver())

def config_for(thread_id: str) -> dict:
    if not thread_id.strip():
        raise ValueError("대화 ID가 필요합니다.")
    return {"configurable": {"thread_id": thread_id}}

def send(graph, thread_id: str, question: str):
    if not question.strip():
        raise ValueError("질문이 비어 있습니다.")
    update = {"messages": [HumanMessage(content=question, id=str(uuid4()))]}
    return graph.invoke(update, config=config_for(thread_id))

graph = make_graph()
reply = send(graph, 'A', '대출 기간?')
print(reply['messages'][-1].content)

이 대화의 사용자 발화 1개. 최근 질문: 대출 기간?


사용자 발화가 1개로 나옵니다. respond가 이 대화 안의 HumanMessage만 세므로 AI 답변은 개수에 들어가지 않습니다.

대화 A에서 두 번째 질문을 보낼 때 이전 발화를 직접 다시 보내지 않아도 개수가 늘어납니다. 체크포인터가 A의 기록을 찾아 이어가기 때문입니다. 같은 두 번째 질문을 새 thread_id로 보내면 첫 발화로 세는지 확인해 보세요.


### 그래프·체크포인터·스레드의 역할

<p>add_messages는 새 메시지를 기존 목록에 어떻게 반영할지 정합니다. 체크포인터는 대화별 그래프 상태를 저장하고 다음 실행에서 이어 쓰게 합니다. thread_id는 어느 대화의 상태를 사용할지 지정하는 이름입니다. 같은 이름을 써도 다른 저장소 객체를 만들면 이전 기록을 찾을 수 없습니다.</p>

<table><caption>역할 비교</caption><thead><tr><th scope="col">요소</th><th scope="col">질문</th><th scope="col">이 예제</th></tr>
</thead><tbody><tr><td>messages 갱신 규칙</td><td>새 메시지를 어떻게 합칠까?</td><td>add_messages</td></tr>
<tr><td>체크포인터</td><td>대화 상태를 어디에 보관할까?</td><td>InMemorySaver의 RAM</td></tr>
<tr><td>thread_id</td><td>어느 대화를 이어갈까?</td><td>A / B / C</td></tr>
</tbody></table>


### 체크포인터를 지정한 그래프 재사용

<p>make_graph는 호출할 때 InMemorySaver를 새로 만듭니다. 예제의 시작에서 한 번만 호출해 graph를 만든 뒤 여러 send 호출에 같은 객체를 넘깁니다. 질문을 보낼 때마다 make_graph를 다시 호출하면 그때마다 새 저장소가 생겨 이전 상태가 이어지지 않습니다.</p>

<p>RAM에 있는 상태는 프로그램이 종료되면 사라집니다. 이 동작을 데이터베이스 영구 저장으로 설명하지 마세요. 후반 서비스에서는 보관 기간·삭제·접근 제어까지 별도 설계합니다. 여기서는 저장소 수명과 대화 구분의 원리를 확인합니다.</p>


### A의 두 번째 invoke 전에 어떤 상태가 복원될까요?

같은 graph와 thread_id를 다시 사용합니다. 첫 질문과 응답을 복원한 뒤 이번 사용자 질문만 더해 respond에 전달합니다.

1. A 선택 · thread_id = A
2. 상태 복원 · 직전 messages = 사용자 1 · 응답 1
3. 입력 병합 · 사용자 발화 수 = 2
4. respond · 최근 질문 = 반납 방법?
5. 상태 보관 · A의 messages = 사용자 2 · 응답 2


### 셀 02 · 같은 thread_id의 체크포인트

이미 저장된 첫 질문을 다시 넘기지 말고 두 번째 질문만 보냅니다.


In [2]:
reply = send(graph, 'A', '반납 방법?')
print(reply['messages'][-1].content)

이 대화의 사용자 발화 2개. 최근 질문: 반납 방법?


사용자 발화 2개가 되었습니다. 같은 저장소와 같은 A를 썼기 때문입니다. 중간에 첫 셀을 다시 실행하면 저장소를 새로 만들어 과거 상태가 사라집니다.


### 동일 thread_id의 연속 입력

<p>thread_id는 State의 messages 안에 넣지 않고 호출 config의 configurable에 넣습니다. send는 이번 사용자 질문 한 개만 변경분으로 보냅니다. 체크포인터가 가져온 이전 상태에 add_messages 규칙으로 새 입력이 반영되고 respond가 실행됩니다.</p>

<p>이전 전체 기록을 새로운 ID로 다시 만들어 보내면 이전 질문이 중복 추가될 수 있습니다. 새 질문과 이미 저장된 이력의 책임을 나눕니다. 이 장은 수정·삭제 요청을 구현하지 않으며 새 입력 추가만 다룹니다.</p>


## thread_id를 이용한 대화 분리

학생 두 명에게 같은 대화 ID를 주면 기록이 섞일 수 있습니다. 새 대화 ID와 기존 대화 ID는 서로 다른 상태를 가리킵니다. InMemorySaver의 기록은 현재 프로세스가 끝나면 사라집니다.

### 같은 저장소에서 대화 A와 B 따라가기

**A에 대출 기간을 묻습니다.**　A에 쌓인 사용자 발화는 한 개입니다.

```
A · 1개 · 대출 기간?
```

**A에서 반납 방법을 묻습니다.**　같은 대화를 다시 열었으므로 첫 질문도 기록에 남습니다.

```
A · 2개 · 반납 방법?
```

**B에서 휴관일을 묻습니다.**　같은 저장소를 써도 B의 사용자 발화는 한 개입니다. A의 두 발화는 A에 남습니다.

```
A · 2개 · 대출 기간? / 반납 방법?
B · 1개 · 휴관일?
```


### 과제 03 · 다른 스레드의 메시지 수

B 대화에 ‘휴관일?’을 보내 마지막 답의 발화 수를 확인하세요.

A의 두 번의 왕복은 사용자 → 답변 → 사용자 → 답변 순입니다. 전체 메시지는 4개, 그중 사용자 질문은 2개입니다. B에서 첫 질문 수가 1이 되어도 A의 네 메시지는 같은 저장소의 다른 ID 아래 남아 있습니다.


In [3]:
# graph는 그대로 사용하고 대화 ID만 B로 바꾸세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>reply_b = send(graph, &#x27;B&#x27;, &#x27;휴관일?&#x27;)
print(reply_b[&#x27;messages&#x27;][-1].content)</code></pre>
<p>실행 결과</p><pre><code>이 대화의 사용자 발화 1개. 최근 질문: 휴관일?</code></pre>
<p>A에 두 번 질문했어도 B의 첫 질문은 1개입니다. B에서 3개라면 호출 때 전달한 ID부터 살펴보세요.</p>
</details>


### 두 번째 대화가 읽는 기록

<p>graph를 그대로 두고 config의 thread_id만 B로 바꾸세요. A의 두 질문을 보관한 저장소를 공유해도 B가 시작하는 메시지 목록에는 그 질문들이 없습니다. 출력의 사용자 발화 수로 구별할 수 있습니다.</p>


### 새 스레드와 저장소의 수명

<table><caption>막혔을 때</caption><thead><tr><th scope="col">증상</th><th scope="col">확인</th><th scope="col">조치</th></tr>
</thead><tbody><tr><td>같은 대화가 이어지지 않음</td><td>graph 객체·thread_id</td><td>같은 graph와 같은 ID인지 비교</td></tr>
<tr><td>다른 대화가 섞임</td><td>A/B에 실제로 전달한 ID</td><td>호출 인자 기록 확인</td></tr>
<tr><td>질문이 중복됨</td><td>전체 이력 재전송·새 ID</td><td>이번 새 사용자 입력만 전달</td></tr>
<tr><td>프로그램 재실행 후 기록 없음</td><td>InMemorySaver 수명</td><td>RAM 저장의 정상 범위로 기록</td></tr>
<tr><td>빈 대화 ID 오류</td><td>config_for 입력</td><td>비어 있지 않은 학습용 ID 사용</td></tr>
</tbody></table>


## 돌아보기

graph.get_state(config_for('A')).values에서 메시지 수를 예측해 보세요. 두 발화에 각각 답이 붙어 총 네 개입니다. 이미 실행한 셀을 반복하면 그만큼 새 발화가 더해집니다.
